In [9]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, to_timestamp, hour, dayofweek
from pyspark.sql.types import DoubleType
from pyspark.ml.feature import VectorAssembler, StandardScaler, StringIndexer
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator,
    RegressionEvaluator,
    ClusteringEvaluator
)
import matplotlib.pyplot as plt

spark = SparkSession.builder \
    .appName("Day7Project") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("WARN")
print(f"Spark version: {spark.version}")
#If you get the "python3 not found" error, add before the above:

import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

Spark version: 4.2.0


In [26]:

# DOMAIN 1: HEALTHCARE - Heart Disease (Fixed)

import math 

print("\n--- Running Domain 1: Healthcare ---")

df = spark.read.csv("day7_data/heart_disease.csv", header=True, inferSchema=True)

clean = df.dropna()
clean = clean.withColumn("target", col("target").cast(DoubleType()))
clean = clean.withColumn("label", when(col("target") > 0, 1.0).otherwise(0.0))
clean = clean.drop("target")

feature_cols = ["age","sex","cp","trestbps","chol","fbs","restecg",
                "thalach","exang","oldpeak","slope","ca","thal"]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="raw_features")
assembled = assembler.transform(clean)
scaler = StandardScaler(inputCol="raw_features", outputCol="features", withMean=True, withStd=True)
final = scaler.fit(assembled).transform(assembled).select("features", "label")

train, test = final.randomSplit([0.8, 0.2], seed=42)
lr = LogisticRegression(featuresCol="features", labelCol="label", maxIter=100)
model = lr.fit(train)

predictions = model.transform(test)

acc = MulticlassClassificationEvaluator(labelCol="label", metricName="accuracy").evaluate(predictions)
auc = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC").evaluate(predictions)

print(f"\n Accuracy: {acc:.4f}")
print(f" AUC: {auc:.4f}")

print("\nConfusion Matrix (Label vs Prediction):")
predictions.groupBy("label", "prediction").count().orderBy("label", "prediction").show()

importance = list(zip(feature_cols, model.coefficients.toArray()))
importance.sort(key=lambda x: math.fabs(x[1]), reverse=True)

print("\nTop 5 Most Important Features:")
for name, coef in importance[:5]:
    print(f"  {name:10s} {coef:+.4f}")


--- Running Domain 1: Healthcare ---

 Accuracy: 0.8723
 AUC: 0.8907

Confusion Matrix (Label vs Prediction):
+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|  0.0|       0.0|   26|
|  0.0|       1.0|    1|
|  1.0|       0.0|    5|
|  1.0|       1.0|   15|
+-----+----------+-----+


Top 5 Most Important Features:
  ca         +1.2143
  thal       +0.6565
  cp         +0.5910
  thalach    -0.5586
  sex        +0.5445


In [15]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.ml.feature import *
from pyspark.ml.classification import *
from pyspark.ml.regression import *
from pyspark.ml.clustering import *
from pyspark.ml.evaluation import *
from pyspark.ml import Pipeline

spark = SparkSession.builder \
    .appName("Day7_All_Domains") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("WARN")
print(" Spark Initialized Successfully!")

 Spark Initialized Successfully!


In [20]:

# DOMAIN 2: FINANCE - Credit Card Fraud

print("\n--- Running Domain 2: Finance ---")

try:
  
    df_cc = spark.read.csv("day7_data/creditcard.csv", header=True, inferSchema=True)
    
    print(f"Total Rows: {df_cc.count()}")
    print("Class Distribution (0 = Normal, 1 = Fraud):")
    df_cc.groupBy("Class").count().show()
 
    clean_cc = df_cc.withColumnRenamed("Class", "label")
    clean_cc = clean_cc.withColumn("label", col("label").cast(DoubleType()))
 
    feature_cols_cc = [f"V{i}" for i in range(1, 29)] + ["Amount"]
  
    assembler_cc = VectorAssembler(inputCols=feature_cols_cc, outputCol="raw_features")
    scaler_cc = StandardScaler(inputCol="raw_features", outputCol="features", withMean=True, withStd=True)
    pipeline_cc = Pipeline(stages=[assembler_cc, scaler_cc])
    
    final_cc = pipeline_cc.fit(clean_cc).transform(clean_cc).select("features", "label")
    
  
    train_cc, test_cc = final_cc.randomSplit([0.8, 0.2], seed=42)
   
    rf_cc = RandomForestClassifier(featuresCol="features", labelCol="label", numTrees=50, maxDepth=10, seed=42)
    
    print("Training Random Forest model (this may take a minute)...")
    model_cc = rf_cc.fit(train_cc)
    
    preds_cc = model_cc.transform(test_cc)
    
 
    auc_cc = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC").evaluate(preds_cc)
    
    print("\n" + "="*50)
    print(f" Finance -> AUC: {auc_cc:.4f}")
    print("="*50)
    
    print("\nConfusion Matrix (Label vs Prediction):")
    preds_cc.groupBy("label", "prediction").count().orderBy("label", "prediction").show()

except Exception as e:
    print(f"\n Error: Make sure 'creditcard.csv' is inside the 'day7_data' folder.")
    print(f"Error details: {e}")


--- Running Domain 2: Finance ---
Total Rows: 284807
Class Distribution (0 = Normal, 1 = Fraud):
+-----+------+
|Class| count|
+-----+------+
|    1|   492|
|    0|284315|
+-----+------+

Training Random Forest model (this may take a minute)...

 Finance -> AUC: 0.9724

Confusion Matrix (Label vs Prediction):
+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|  0.0|       0.0|56908|
|  0.0|       1.0|    9|
|  1.0|       0.0|   26|
|  1.0|       1.0|   72|
+-----+----------+-----+



In [22]:

# DOMAIN 3: RETAIL - Customer Segmentation
print("\n--- Running Domain 3: Retail ---")

try:
  
    df_retail = spark.read.csv("day7_data/online_retail.csv", header=True, inferSchema=True)
    
    clean_retail = df_retail.filter(
        (col("CustomerID").isNotNull()) & 
        (col("Quantity") > 0) & 
        (col("UnitPrice") > 0)
    )
  
    clean_retail = clean_retail.withColumn("TotalPrice", col("Quantity") * col("UnitPrice"))
    
 
    snapshot = clean_retail.agg(max("InvoiceDate")).collect()[0][0]
    
    rfm = clean_retail.groupBy("CustomerID").agg(
        datediff(lit(snapshot), max("InvoiceDate")).alias("Recency"),
        count("InvoiceNo").alias("Frequency"),
        sum("TotalPrice").alias("Monetary")
    )
   
    assembler_retail = VectorAssembler(inputCols=["Recency", "Frequency", "Monetary"], outputCol="raw_features")
    scaler_retail = StandardScaler(inputCol="raw_features", outputCol="features", withMean=True, withStd=True)
    pipeline_retail = Pipeline(stages=[assembler_retail, scaler_retail])
    
    final_retail = pipeline_retail.fit(rfm).transform(rfm)
   
    kmeans = KMeans(featuresCol="features", k=4, seed=42)
    model_retail = kmeans.fit(final_retail)
   
    clustered = model_retail.transform(final_retail)
    
    silhouette = ClusteringEvaluator(featuresCol="features").evaluate(clustered)
    
    print("\n" + "="*50)
    print(f" Retail -> Silhouette Score: {silhouette:.4f}")
    print("="*50)
    
    print("\nCustomer Distribution per Cluster:")
    clustered.groupBy("prediction").count().orderBy("prediction").show()
    
    print("\nAverage RFM per Cluster (Interpretation):")
    clustered.groupBy("prediction").agg(
        round(avg("Recency"), 1).alias("Avg_Recency"),
        round(avg("Frequency"), 1).alias("Avg_Frequency"),
        round(avg("Monetary"), 2).alias("Avg_Monetary")
    ).orderBy("prediction").show()

except Exception as e:
    print(f"\n Error: Make sure 'online_retail.csv' is inside the 'day7_data' folder.")
    print(f"Error details: {e}")


--- Running Domain 3: Retail ---

 Retail -> Silhouette Score: 0.7291

Customer Distribution per Cluster:
+----------+-----+
|prediction|count|
+----------+-----+
|         0| 3248|
|         1| 1080|
|         2|    4|
|         3|    6|
+----------+-----+


Average RFM per Cluster (Interpretation):
+----------+-----------+-------------+------------+
|prediction|Avg_Recency|Avg_Frequency|Avg_Monetary|
+----------+-----------+-------------+------------+
|         0|       40.9|        104.6|     2092.09|
|         1|      246.8|         27.7|       636.5|
|         2|        1.5|       5807.0|    70925.29|
|         3|        7.0|        826.8|   190863.46|
+----------+-----------+-------------+------------+



In [23]:

# DOMAIN 4: SMART CITIES - Traffic Volume

print("\n--- Running Domain 4: Smart Cities ---")

try:
    
    df_traffic = spark.read.csv("day7_data/traffic.csv", header=True, inferSchema=True)
    
    print(f"Total Rows: {df_traffic.count()}")
 
    # Convert the date_time string into a timestamp, then extract the hour and day of the week
    df_traffic = df_traffic.withColumn("date_time", to_timestamp("date_time"))
    df_traffic = df_traffic.withColumn("hour", hour("date_time"))
    df_traffic = df_traffic.withColumn("dow", dayofweek("date_time"))
 
    feature_cols_traffic = ["temp", "rain_1h", "clouds_all", "hour", "dow"]
    
    assembler_traffic = VectorAssembler(inputCols=feature_cols_traffic, outputCol="features")
    
    final_traffic = assembler_traffic.transform(df_traffic).select("features", col("traffic_volume").alias("label"))
  
    train_traffic, test_traffic = final_traffic.randomSplit([0.8, 0.2], seed=42)
 
    print("Training Random Forest Regressor (this may take a minute)...")
    rf_traffic = RandomForestRegressor(featuresCol="features", labelCol="label", numTrees=50, maxDepth=10, seed=42)
    model_traffic = rf_traffic.fit(train_traffic)
 
    preds_traffic = model_traffic.transform(test_traffic)
  
    rmse = RegressionEvaluator(labelCol="label", metricName="rmse").evaluate(preds_traffic)
    r2 = RegressionEvaluator(labelCol="label", metricName="r2").evaluate(preds_traffic)
    
    print("\n" + "="*50)
    print(f" Smart Cities -> RMSE: {rmse:.2f}")
    print(f" Smart Cities -> R²: {r2:.4f}")
    print("="*50)
    
    print("\nSample Predictions (Label vs Prediction):")
    preds_traffic.select("label", "prediction").show(5)

except Exception as e:
    print(f"\n Error: Make sure 'traffic.csv' is inside the 'day7_data' folder.")
    print(f"Error details: {e}")


--- Running Domain 4: Smart Cities ---
Total Rows: 48204
Training Random Forest Regressor (this may take a minute)...

 Smart Cities -> RMSE: 546.13
 Smart Cities -> R²: 0.9250

Sample Predictions (Label vs Prediction):
+-----+------------------+
|label|        prediction|
+-----+------------------+
|  284| 560.0911094367214|
|  739|1029.4063379020072|
|  962|  1296.91246237714|
|  354|481.73854160551605|
| 1909|1954.2978066405544|
+-----+------------------+
only showing top 5 rows


In [27]:

# DOMAIN 5: CYBERSECURITY - Intrusion Detection (Fixed)

print("\n--- Running Domain 5: Cybersecurity ---")

try:
    
    columns_kdd = [
        "duration","protocol_type","service","flag","src_bytes","dst_bytes","land",
        "wrong_fragment","urgent","hot","num_failed_logins","logged_in","num_compromised",
        "root_shell","su_attempted","num_root","num_file_creations","num_shells",
        "num_access_files","num_outbound_cmds","is_host_login","is_guest_login",
        "count","srv_count","serror_rate","srv_serror_rate","rerror_rate",
        "srv_rerror_rate","same_srv_rate","diff_srv_rate","srv_diff_host_rate",
        "dst_host_count","dst_host_srv_count","dst_host_same_srv_rate",
        "dst_host_diff_srv_rate","dst_host_same_src_port_rate",
        "dst_host_srv_diff_host_rate","dst_host_serror_rate","dst_host_srv_serror_rate",
        "dst_host_rerror_rate","dst_host_srv_rerror_rate","label","difficulty"
    ]

    df_kdd = spark.read.csv("day7_data/KDDTrain+.txt", header=False, inferSchema=True)
    
    for i, name in enumerate(columns_kdd):
        df_kdd = df_kdd.withColumnRenamed(f"_c{i}", name)

    df_kdd = df_kdd.withColumn("label", when(col("label") == "normal", 0.0).otherwise(1.0))

    cat_cols_kdd = ["protocol_type", "service", "flag"]
    indexers = [StringIndexer(inputCol=c, outputCol=c+"_idx", handleInvalid="keep") for c in cat_cols_kdd]
    num_cols_kdd = [c for c in columns_kdd if c not in cat_cols_kdd + ["label", "difficulty"]]
    
    assembler_kdd = VectorAssembler(
        inputCols=[c+"_idx" for c in cat_cols_kdd] + num_cols_kdd, 
        outputCol="features"
    )
    
    pipeline_kdd = Pipeline(stages=indexers + [assembler_kdd])
    prepared_kdd = pipeline_kdd.fit(df_kdd).transform(df_kdd).select("features", "label")

    train_kdd, test_kdd = prepared_kdd.randomSplit([0.8, 0.2], seed=42)

    print("Training Random Forest Classifier (this may take a minute)...")
    
    rf_kdd = RandomForestClassifier(
        featuresCol="features", 
        labelCol="label", 
        numTrees=50, 
        maxDepth=10, 
        maxBins=100,  
        seed=42
    )
    
    model_kdd = rf_kdd.fit(train_kdd)

    preds_kdd = model_kdd.transform(test_kdd)

    acc_kdd = MulticlassClassificationEvaluator(labelCol="label", metricName="accuracy").evaluate(preds_kdd)
    auc_kdd = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC").evaluate(preds_kdd)
    
    print("\n" + "="*50)
    print(f" Cybersecurity -> Accuracy: {acc_kdd:.4f}")
    print(f" Cybersecurity -> AUC: {auc_kdd:.4f}")
    print("="*50)
    
    print("\nConfusion Matrix (Label vs Prediction):")
    preds_kdd.groupBy("label", "prediction").count().orderBy("label", "prediction").show()

except Exception as e:
    print(f"\n Error: {e}")


--- Running Domain 5: Cybersecurity ---
Training Random Forest Classifier (this may take a minute)...

 Cybersecurity -> Accuracy: 0.9982
 Cybersecurity -> AUC: 1.0000

Confusion Matrix (Label vs Prediction):
+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|  0.0|       0.0|13470|
|  0.0|       1.0|    5|
|  1.0|       0.0|   40|
|  1.0|       1.0|11519|
+-----+----------+-----+



In [28]:

# DOMAIN 6: MANUFACTURING - Predictive Maintenance

print("\n--- Running Domain 6: Manufacturing ---")

try:
 
    df_ai4i = spark.read.csv("day7_data/ai4i2020.csv", header=True, inferSchema=True)
    
    print(f"Total Rows: {df_ai4i.count()}")
   
    df_ai4i = df_ai4i.withColumnRenamed("Machine failure", "label")
    df_ai4i = df_ai4i.withColumn("label", col("label").cast(DoubleType()))
    
    print("Target class distribution (0 = No Failure, 1 = Failure):")
    df_ai4i.groupBy("label").count().show()
   
    feature_cols_ai4i = [
        "Air temperature [K]", 
        "Process temperature [K]", 
        "Rotational speed [rpm]", 
        "Torque [Nm]", 
        "Tool wear [min]"
    ]
    
    assembler_ai4i = VectorAssembler(inputCols=feature_cols_ai4i, outputCol="raw_features")
    assembled_ai4i = assembler_ai4i.transform(df_ai4i)
    
    scaler_ai4i = StandardScaler(inputCol="raw_features", outputCol="features", withMean=True, withStd=True)
    final_ai4i = scaler_ai4i.fit(assembled_ai4i).transform(assembled_ai4i).select("features", "label")
   
    train_ai4i, test_ai4i = final_ai4i.randomSplit([0.8, 0.2], seed=42)
    
    print("Training Random Forest Classifier...")
    rf_ai4i = RandomForestClassifier(featuresCol="features", labelCol="label", numTrees=50, maxDepth=10, seed=42)
    model_ai4i = rf_ai4i.fit(train_ai4i)
   
    preds_ai4i = model_ai4i.transform(test_ai4i)
   
    auc_ai4i = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC").evaluate(preds_ai4i)
    
    print("\n" + "="*50)
    print(f" Manufacturing -> AUC: {auc_ai4i:.4f}")
    print("="*50)
    
    print("\nConfusion Matrix (Label vs Prediction):")
    preds_ai4i.groupBy("label", "prediction").count().orderBy("label", "prediction").show()

except Exception as e:
    print(f"\n Error: Make sure 'ai4i2020.csv' is inside the 'day7_data' folder.")
    print(f"Error details: {e}")


--- Running Domain 6: Manufacturing ---
Total Rows: 10000
Target class distribution (0 = No Failure, 1 = Failure):
+-----+-----+
|label|count|
+-----+-----+
|  0.0| 9661|
|  1.0|  339|
+-----+-----+

Training Random Forest Classifier...

 Manufacturing -> AUC: 0.9598

Confusion Matrix (Label vs Prediction):
+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|  0.0|       0.0| 1842|
|  0.0|       1.0|   12|
|  1.0|       0.0|   38|
|  1.0|       1.0|   29|
+-----+----------+-----+

